# Flipkart GridLock Hackathon: Traffic Demand Prediction
**Author**: Pair Programming with Antigravity AI

This notebook contains a complete, production-grade machine learning pipeline to solve the traffic demand forecasting challenge.

### Solution Architecture Overview:
1. **Spatial Features**: Decode `geohash` codes to physical Latitude/Longitude coordinates.
2. **Temporal Features**: Convert time string timestamps into continuous minutes-from-midnight and cyclical sine/cosine representations.
3. **Historical Demand Matching**: Extract the exact matching spatial-temporal traffic demand from the previous full day (Day 48).
4. **Active Trend Deviation**: Compute the localized traffic volume growth/decline ratio between Day 49 and Day 48 morning hours to capture holiday or weather effects.
5. **5-Fold Cross-Validation**: Train LightGBM regressors across folds to produce robust, variance-reduced ensembled predictions.

## 1. Setup & Dependencies

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as gh
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import os
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
print('Dependencies loaded successfully!')

## 2. Advanced Feature Engineering & Preprocessing
Here we define our advanced preprocessing logic, which extracts structural spatial, temporal, and historical features:

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as gh

def load_and_preprocess_data(train_path="dataset/train.csv", test_path="dataset/test.csv"):
    print("Loading data...")
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    
    # Identify test indices for splitting back later
    train['is_test'] = 0
    test['is_test'] = 1
    test['demand'] = np.nan
    
    # Combine datasets for unified preprocessing
    full_df = pd.concat([train, test], ignore_index=True)
    
    # 1. Temporal features
    print("Parsing timestamps...")
    def ts_to_minutes(ts):
        h, m = map(int, ts.split(':'))
        return h * 60 + m
    
    full_df['minutes'] = full_df['timestamp'].apply(ts_to_minutes)
    full_df['hour'] = full_df['minutes'] / 60.0
    
    # Cyclical trigonometric features for time of day
    full_df['sin_time'] = np.sin(2 * np.pi * full_df['minutes'] / 1440.0)
    full_df['cos_time'] = np.cos(2 * np.pi * full_df['minutes'] / 1440.0)
    
    # 2. Geohash decoding (Spatial features)
    print("Decoding geohashes...")
    # Cache geohash coordinates to speed up computation
    unique_geohashes = full_df['geohash'].unique()
    geohash_coords = {}
    for g in unique_geohashes:
        try:
            lat, lon = gh.decode(g)
            geohash_coords[g] = (lat, lon)
        except Exception:
            geohash_coords[g] = (np.nan, np.nan)
            
    full_df['latitude'] = full_df['geohash'].map(lambda g: geohash_coords[g][0])
    full_df['longitude'] = full_df['geohash'].map(lambda g: geohash_coords[g][1])
    
    # 3. Previous Day Demand Lookup (demand_last_day)
    print("Creating historical lag features...")
    # Group train day 48 demand by geohash and timestamp
    df_48 = train[train['day'] == 48]
    day48_lookup = df_48.groupby(['geohash', 'timestamp'])['demand'].mean().reset_index()
    day48_lookup.rename(columns={'demand': 'demand_last_day'}, inplace=True)
    
    # Merge back to full_df
    full_df = pd.merge(full_df, day48_lookup, on=['geohash', 'timestamp'], how='left')
    
    # 4. Day 49 Morning Trend Ratio
    # Compute trend of demand on day 49 compared to day 48 between 00:00 and 02:00
    print("Computing Day 49 morning trend ratio...")
    morning_timestamps = ['0:0', '0:15', '0:30', '0:45', '1:0', '1:15', '1:30', '1:45', '2:0']
    
    df_48_morning = train[(train['day'] == 48) & (train['timestamp'].isin(morning_timestamps))]
    df_49_morning = train[(train['day'] == 49) & (train['timestamp'].isin(morning_timestamps))]
    
    mean_48 = df_48_morning.groupby('geohash')['demand'].mean()
    mean_49 = df_49_morning.groupby('geohash')['demand'].mean()
    
    trend_ratio = mean_49 / (mean_48 + 1e-5)
    trend_ratio = trend_ratio.rename('day49_trend_ratio')
    
    # Map trend ratio back to full_df
    full_df = pd.merge(full_df, trend_ratio, on='geohash', how='left')
    
    # For Day 48 rows, set trend ratio to 1.0 (since there is no comparison)
    # For day 49 rows without prior morning data, fill NaN with 1.0 (no change)
    full_df.loc[full_df['day'] == 48, 'day49_trend_ratio'] = 1.0
    full_df['day49_trend_ratio'] = full_df['day49_trend_ratio'].fillna(1.0)
    
    # Clip extreme ratios to prevent huge outlier swings
    full_df['day49_trend_ratio'] = full_df['day49_trend_ratio'].clip(0.1, 5.0)
    
    # 5. Missing Value Imputation
    print("Imputing missing values...")
    full_df['RoadType'] = full_df['RoadType'].fillna('Missing')
    full_df['Weather'] = full_df['Weather'].fillna('Missing')
    
    # Median temperature per Weather
    weather_temp_medians = full_df.groupby('Weather')['Temperature'].transform('median')
    full_df['Temperature'] = full_df['Temperature'].fillna(weather_temp_medians)
    full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())
    
    # 6. Categorical Encoding
    print("Encoding categorical columns...")
    full_df['LargeVehicles'] = full_df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(0).astype(int)
    full_df['Landmarks'] = full_df['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    
    # Label encode categorical columns for LightGBM
    for col in ['RoadType', 'Weather', 'geohash']:
        full_df[col] = full_df[col].astype('category')
        
    # Split back into train and test
    train_preprocessed = full_df[full_df['is_test'] == 0].drop(columns=['is_test', 'Index']).reset_index(drop=True)
    test_preprocessed = full_df[full_df['is_test'] == 1].drop(columns=['is_test', 'demand']).reset_index(drop=True)
    
    # Define features to use
    features = [
        'latitude', 'longitude', 'minutes', 'hour',
        'sin_time', 'cos_time', 'demand_last_day', 'day49_trend_ratio',
        'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
        'Temperature', 'Weather', 'geohash'
    ]
    
    target = 'demand'
    
    print(f"Preprocessing completed. Train shape: {train_preprocessed.shape}, Test shape: {test_preprocessed.shape}")
    return train_preprocessed, test_preprocessed, features, target

# Verification run
train_df, test_df, features, target = load_and_preprocess_data()
print('\nFeatures for training:', features)

## 3. 5-Fold Cross-Validation and Model Training
We set up a robust Out-of-Fold (OOF) cross-validation framework to prevent data leakage and train 5 LightGBM Regressors. CatBoost and XGBoost hyperparameters can also be optimized here.

In [ ]:
def train_model():
    # Load and preprocess data
    train_df, test_df, features, target = load_and_preprocess_data()
    
    print("\n--- Starting Training ---")
    print(f"Features used ({len(features)}): {features}")
    
    # Initialize variables for Out-Of-Fold (OOF) predictions
    oof_predictions = np.zeros(len(train_df))
    test_predictions = np.zeros(len(test_df))
    
    # 5-Fold Cross Validation
    n_splits = 5
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    models = []
    oof_rmses = []
    oof_maes = []
    
    os.makedirs("models", exist_ok=True)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        print(f"\n--- Fold {fold + 1} / {n_splits} ---")
        
        X_train, y_train = train_df.loc[train_idx, features], train_df.loc[train_idx, target]
        X_val, y_val = train_df.loc[val_idx, features], train_df.loc[val_idx, target]
        
        # Define LightGBM dataset
        train_data = lgb.Dataset(X_train, label=y_train)
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
        
        # Hyperparameters for LightGBM Regressor
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'n_estimators': 3000,
            'learning_rate': 0.05,
            'num_leaves': 63,
            'max_depth': 8,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': 42 + fold,
            'verbose': -1,
            'n_jobs': -1
        }
        
        # Train model with early stopping
        model = lgb.train(
            params,
            train_data,
            valid_sets=[train_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=200)
            ]
        )
        
        # Predict on validation fold
        val_preds = model.predict(X_val, num_iteration=model.best_iteration)
        oof_predictions[val_idx] = val_preds
        
        # Evaluate fold performance
        fold_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        fold_mae = mean_absolute_error(y_val, val_preds)
        oof_rmses.append(fold_rmse)
        oof_maes.append(fold_mae)
        
        print(f"Fold {fold + 1} RMSE: {fold_rmse:.6f} | MAE: {fold_mae:.6f}")
        
        # Predict on test set (average predictions across all folds)
        test_preds = model.predict(test_df[features], num_iteration=model.best_iteration)
        test_predictions += test_preds / n_splits
        
        # Save model
        model_path = f"models/lgb_fold_{fold + 1}.pkl"
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
            
    # Final Out-Of-Fold Evaluation
    overall_rmse = np.sqrt(mean_squared_error(train_df[target], oof_predictions))
    overall_mae = mean_absolute_error(train_df[target], oof_predictions)
    overall_r2 = r2_score(train_df[target], oof_predictions)
    
    print("\n--- Out-Of-Fold Summary ---")
    print(f"Mean Fold RMSE: {np.mean(oof_rmses):.6f} ± {np.std(oof_rmses):.6f}")
    print(f"Overall OOF RMSE: {overall_rmse:.6f}")
    print(f"Overall OOF MAE: {overall_mae:.6f}")
    print(f"Overall OOF R2 Score: {overall_r2:.6f}")
    
    # Save the test predictions for quick submission generation
    print("\nSaving baseline predictions...")
    test_df_out = test_df.copy()
    test_df_out['demand'] = test_predictions
    
    # Clip demand to be non-negative (since traffic demand cannot be less than 0)
    test_df_out['demand'] = test_df_out['demand'].clip(0.0, 1.0)
    
    os.makedirs("predictions", exist_ok=True)
    test_df_out.to_csv("predictions/test_predictions.csv", index=False)
    
    # Generate submission file
    submission = test_df_out[['Index', 'demand']]
    submission.to_csv("submission.csv", index=False)
    print("Baseline submission.csv generated successfully!")
    
    return oof_predictions, test_predictions

# Run the complete training loop!
oof_preds, test_preds = train_model()

## 4. Model Interpretation & Feature Importance
Let's look at which features are most predictive for traffic demand forecasting:

In [ ]:
# Load one of the trained fold models to plot feature importance
with open('models/lgb_fold_1.pkl', 'rb') as f:
    model = pickle.load(f)

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'], color='teal')
plt.xlabel('Gain Importance')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()

## 5. Final Submission Verification
Let's verify that the output submission file meets all format requirements (correct row count, non-negative demand values, non-null values):

In [ ]:
sub = pd.read_csv('submission.csv')
print('--- Submission Head ---')
print(sub.head())

print('\n--- Submission Info ---')
print(sub.info())

print('\nNull count:', sub.isnull().sum().sum())
print('Min demand:', sub['demand'].min())
print('Max demand:', sub['demand'].max())
print('Number of rows matching test set:', len(sub) == 41778)